# Accelerated Weathering of Limestone (AWL) Ship Reactor Box Model
## Dong et al. (2025), *Science Advances*

This notebook implements the **Accelerated Weathering of Limestone (AWL)** box model.  
The reactor dissolves limestone in CO₂-rich seawater aboard cargo ships, converting flue gas CO₂  
to stable bicarbonate ions to achieve permanent ocean carbon sequestration.

> **Reference:** Dong et al. (2025). *Science Advances*, eadr7250.  
> **Kinetics:** Naviaux et al. (2019). *GCA* 246:363-384.


## 1. Imports

In [1]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
import plotly.io as pio
pio.renderers.default = 'browser'

## 2. Carbonate System Solver

Calculates seawater pH, calcite saturation state (Ω), pCO₂, and dissolved CO₂ from alkalinity and DIC.

> **Note:** Uses simplified equilibrium constants at T=25°C, S=35.  
> For quantitative agreement with the paper, replace with `PyCO2SYS`.


In [3]:
def calc_carbonate_system(Alk_mol_m3, DIC_mol_m3, T_K=298.15, S=35.0):
    """
    Simplified carbonate system calculation.
    Returns pH, Omega_calcite, pCO2 (atm), [CO2] (mol/m3).
    
    Uses simplified equilibrium constants appropriate for seawater.
    For production use, replace with PyCO2SYS.
    
    Args:
        Alk_mol_m3: Total alkalinity in mol/m3
        DIC_mol_m3: Dissolved inorganic carbon in mol/m3
        T_K: Temperature in Kelvin (default 298.15 = 25 C)
        S: Salinity (default 35)
    
    Returns:
        pH, Omega_calcite, pCO2_atm, CO2aq_mol_m3
    """
    # Convert to mol/kg (approximate: rho_sw ~ 1025 kg/m3)
    rho_sw = 1025.0  # kg/m3
    Alk = Alk_mol_m3 / rho_sw * 1e6  # umol/kg
    DIC = DIC_mol_m3 / rho_sw * 1e6  # umol/kg

    # Equilibrium constants (Mehrbach refitted by Dickson & Millero 1987)
    # K1, K2 in mol/kg-sw
    K1 = 10**(-3.561)  # approximate for T=25C, S=35
    K2 = 10**(-9.08)   # approximate for T=25C, S=35
    
    # Henry's constant from paper: H = 29.41 atm*kg/mol
    H = 29.41  # atm*kg/mol
    KH = 1.0 / H  # mol/kg/atm
    
    # Ksp for calcite (mol2/kg2, T=25C, S=35, P=1 atm)
    Ksp_calcite = 4.27e-7  # mol2/kg2
    
    # Ca2+ concentration (mol/kg)
    Ca = 0.01028  # approximately 10.28 mmol/kg in seawater S=35
    
    # Solve for [H+] iteratively
    # Charge balance: Alk = [HCO3-] + 2[CO3--] + [OH-] - [H+]
    # DIC = [CO2*] + [HCO3-] + [CO3--]
    
    Alk_mol = Alk * 1e-6
    DIC_mol = DIC * 1e-6
    
    Kw = 6.3e-14  # Kw at S=35, T=25C
    
    def carbonate_eq(pH):
        h = 10**(-pH)
        # From DIC and equilibrium constants:
        # [CO2*] = DIC / (1 + K1/h + K1*K2/h^2)
        denom = 1.0 + K1/h + K1*K2/h**2
        CO2 = DIC_mol / denom
        HCO3 = K1 * CO2 / h
        CO3 = K2 * HCO3 / h
        OH = Kw / h
        
        Alk_calc = HCO3 + 2*CO3 + OH - h
        return Alk_calc - Alk_mol
    
    # Bracket the pH solution
    try:
        pH = brentq(carbonate_eq, 4.0, 9.0, xtol=1e-6)
    except ValueError:
        pH = 7.0  # fallback
    
    h = 10**(-pH)
    denom = 1.0 + K1/h + K1*K2/h**2
    CO2_mol_kg = DIC_mol / denom
    HCO3_mol_kg = K1 * CO2_mol_kg / h
    CO3_mol_kg = K2 * HCO3_mol_kg / h
    
    # Omega calcite
    Omega = Ca * CO3_mol_kg / Ksp_calcite
    
    # pCO2 from Henry's law
    pCO2_atm = CO2_mol_kg / KH
    
    # [CO2]aq in mol/m3
    CO2aq_mol_m3 = CO2_mol_kg * rho_sw
    
    return pH, Omega, pCO2_atm, CO2aq_mol_m3

## 3. Calcite Dissolution Kinetics

Two-segment rate law from Naviaux et al. (2019), Eq. 10-11 in the paper:
- Ω < 0.75: Rate = 10⁻⁵·⁸³ × (1 − Ω)⁴·¹⁸  mol m⁻² s⁻¹
- 0.75 ≤ Ω < 1: Rate = 10⁻⁷·⁰⁶ × (1 − Ω)²·¹⁵  mol m⁻² s⁻¹

Grain surface area (Eq. 4): SA = 6 / (ρ_calcite × diameter) × wt_calcite


In [4]:
def calcite_dissolution_rate(Omega):
    """
    Two-segment calcite dissolution rate law (Naviaux et al. 2019).
    
    Rate = 10^-5.83 * (1-Omega)^4.18   for Omega < 0.75
    Rate = 10^-7.06 * (1-Omega)^2.15   for 0.75 < Omega < 1
    Rate = 0                             for Omega >= 1 (no dissolution above saturation)
    
    Returns:
        Rate in mol / (m2 * s)
    """
    if Omega >= 1.0:
        return 0.0
    elif Omega < 0.75:
        return 10**(-5.83) * (1.0 - Omega)**4.18
    else:  # 0.75 <= Omega < 1.0
        return 10**(-7.06) * (1.0 - Omega)**2.15

In [5]:
def surface_area_geom(wt_calcite_kg, diameter_m, rho_calcite=2710.0):
    """
    Geometric surface area of calcite grains (Eq. 4).
    
    SA_geom = 6 / (rho_calcite * diameter) * wt_calcite
    
    Args:
        wt_calcite_kg: Weight of calcite in reactor (kg)
        diameter_m: Mean grain diameter (m)
        rho_calcite: Density of calcite (kg/m3), default 2710
    
    Returns:
        Surface area in m2
    """
    return 6.0 / (rho_calcite * diameter_m) * wt_calcite_kg

In [6]:
def dissolution_flux(Omega, SA_m2):
    """
    Total dissolution flux in mol/s.
    
    F_diss = Rate [mol/m2/s] * SA [m2]
    """
    return calcite_dissolution_rate(Omega) * SA_m2

## 4. Box Model ODE System (Eqs. 5-8)

Mass balance ODEs for the 4-unit PCST reactor.  
State variables per dissolution tank (DT): Alkalinity, DIC, Ca²⁺ (all mol/m³).

**Processes per DT:**
1. Advection: seawater flows in from upstream unit
2. CO₂ solubilization: CO₂ from adjacent absorption column (AC) dissolves into seawater
3. Calcite dissolution: CaCO₃ dissolves driven by undersaturation; adds 2 mol HCO₃⁻ + 1 mol Ca²⁺ per mol dissolved

**Flow modes:** `parallel` (co-current) or `counter` (counterflow — higher efficiency)


In [7]:
def reactor_odes(t, y, params):
    """
    ODE system for the 4-unit PCST reactor.
    
    State vector y has 3 variables per DT (Alk, DIC, Ca) = 12 total.
    
    Units:
        Alk, DIC, Ca in mol/m3
        Fluxes in mol/s
        Volume in m3
        Flow rates in m3/s
    
    Args:
        t: time (s)
        y: state vector [Alk_1, DIC_1, Ca_1, Alk_2, DIC_2, Ca_2, ...]
        params: dict of model parameters
    
    Returns:
        dydt: time derivatives
    """
    n_units = 4
    
    # Unpack parameters
    V_DT = params['V_DT']           # Volume of each DT (m3)
    Fsw = params['Fsw']             # Seawater flow rate (m3/s)
    FFlue = params['FFlue']         # Flue gas flow rate (m3/s)
    pCO2_flue = params['pCO2_flue'] # Flue gas pCO2 (atm)
    SA = params['SA']               # Surface area of calcite per DT (m2)
    Alk_in = params['Alk_in']       # Inflow alkalinity (mol/m3)
    DIC_in = params['DIC_in']       # Inflow DIC (mol/m3)
    Ca_in = params['Ca_in']         # Inflow Ca2+ (mol/m3)
    T_K = params['T_K']             # Temperature (K)
    S = params['S']                 # Salinity
    H = params['H']                 # Henry's constant (atm*kg/mol)
    rho_sw = params['rho_sw']       # Seawater density (kg/m3)
    mode = params['mode']           # 'parallel' or 'counter'
    
    # Reshape state vector
    Alk = y[0::3]   # mol/m3, array of 4 values
    DIC = y[1::3]
    Ca  = y[2::3]
    
    dydt = np.zeros(n_units * 3)
    
    for i in range(n_units):
        # Determine inflow concentrations based on flow mode
        if mode == 'parallel':
            # Parallel: seawater flows 1->2->3->4, gas flows 1->2->3->4
            if i == 0:
                Alk_up = Alk_in
                DIC_up = DIC_in
                Ca_up = Ca_in
            else:
                Alk_up = Alk[i-1]
                DIC_up = DIC[i-1]
                Ca_up = Ca[i-1]
            # Gas pCO2 in AC preceding this DT
            if i == 0:
                pCO2_ac = pCO2_flue  # fresh gas
            else:
                # Simplified: assume gas pCO2 decreases as it absorbs CO2
                # Use current seawater CO2 to estimate equilibration
                pCO2_ac = pCO2_flue * (1.0 - i * 0.15)  # rough approximation
                pCO2_ac = max(pCO2_ac, 0.0)
        
        elif mode == 'counter':
            # Counter: seawater flows 1->2->3->4, gas flows 4->3->2->1
            if i == 0:
                Alk_up = Alk_in
                DIC_up = DIC_in
                Ca_up = Ca_in
            else:
                Alk_up = Alk[i-1]
                DIC_up = DIC[i-1]
                Ca_up = Ca[i-1]
            # In counterflow, seawater in DT_i contacts gas from AC_{n-i}
            # Gas at the end of AC chain (AC4 in counterflow) contacts DT1
            gas_index = (n_units - 1 - i)
            pCO2_ac = pCO2_flue * (1.0 - gas_index * 0.15)
            pCO2_ac = max(pCO2_ac, 0.0)
        
        # [CO2]sat in this DT from Henry's law (Eq. 9)
        # CO2sat = pCO2_ac * rho_sw / H   [mol/m3]
        CO2sat = pCO2_ac * rho_sw / H
        
        # Current [CO2]aq from carbonate system
        _, Omega, _, CO2aq = calc_carbonate_system(Alk[i], DIC[i], T_K, S)
        
        # CO2 solubilization flux from AC: adds to DIC when CO2sat > CO2aq
        # We model AC as achieving complete equilibration (verified in paper)
        # CO2 added = (CO2sat - CO2aq) * Fsw  [mol/s]
        FCO2_solubilization = Fsw * (CO2sat - CO2aq)
        
        # Calcite dissolution flux (Eq. 10-11)
        FDiss = dissolution_flux(Omega, SA)
        
        # ODEs (Eq. 5, 6, 7)
        # dAlk/dt = Fsw * (Alk_up - Alk_i) + 2 * FDiss
        dydt[i*3 + 0] = (Fsw * (Alk_up - Alk[i]) + 2.0 * FDiss) / V_DT
        
        # dDIC/dt = Fsw * (DIC_up - DIC_i) + FCO2_solubilization + FDiss
        dydt[i*3 + 1] = (Fsw * (DIC_up - DIC[i]) + FCO2_solubilization + FDiss) / V_DT
        
        # d[Ca]/dt = Fsw * ([Ca]_up - [Ca]_i) + FDiss
        dydt[i*3 + 2] = (Fsw * (Ca_up - Ca[i]) + FDiss) / V_DT
    
    return dydt

## 5. Simulation Runner and Output Utilities

- `run_reactor_simulation`: integrates ODEs to steady state using LSODA
- `get_steady_state`: extracts final time point
- `compute_efficiencies`: calculates total efficiency (Eq. 1) and conversion efficiency (Eq. 2)


In [8]:
def run_reactor_simulation(params, t_end=7200.0, n_points=500):
    """
    Run the AWL reactor box model to steady state.
    
    Args:
        params: dict of model parameters
        t_end: simulation end time in seconds (default 2 hours)
        n_points: number of output time points
    
    Returns:
        dict with time, Alk, DIC, Ca, pH, Omega, pCO2 for each DT
    """
    n_units = 4
    
    # Initial conditions: all DTs start at inflow seawater composition
    y0 = np.zeros(n_units * 3)
    for i in range(n_units):
        y0[i*3 + 0] = params['Alk_in']
        y0[i*3 + 1] = params['DIC_in']
        y0[i*3 + 2] = params['Ca_in']
    
    t_span = (0, t_end)
    t_eval = np.linspace(0, t_end, n_points)
    
    sol = solve_ivp(
        reactor_odes,
        t_span,
        y0,
        args=(params,),
        t_eval=t_eval,
        method='LSODA',
        rtol=1e-6,
        atol=1e-9
    )
    
    # Extract results
    Alk = sol.y[0::3]  # shape (4, n_points)
    DIC = sol.y[1::3]
    Ca  = sol.y[2::3]
    
    # Calculate derived quantities at each time step for each DT
    pH_all = np.zeros_like(Alk)
    Omega_all = np.zeros_like(Alk)
    pCO2_all = np.zeros_like(Alk)
    
    for i in range(n_units):
        for j in range(len(sol.t)):
            pH, Omega, pCO2, _ = calc_carbonate_system(
                Alk[i,j], DIC[i,j],
                params['T_K'], params['S']
            )
            pH_all[i,j] = pH
            Omega_all[i,j] = Omega
            pCO2_all[i,j] = pCO2
    
    return {
        't': sol.t,
        'Alk': Alk,
        'DIC': DIC,
        'Ca': Ca,
        'pH': pH_all,
        'Omega': Omega_all,
        'pCO2': pCO2_all,
        'success': sol.success
    }

In [9]:
def get_steady_state(results):
    """Extract steady-state values (last time point)."""
    return {
        'Alk': results['Alk'][:, -1],
        'DIC': results['DIC'][:, -1],
        'Ca': results['Ca'][:, -1],
        'pH': results['pH'][:, -1],
        'Omega': results['Omega'][:, -1],
        'pCO2': results['pCO2'][:, -1],
    }

In [10]:
def compute_efficiencies(ss, params):
    """
    Compute instantaneous total efficiency (Eq. 1) and
    conversion efficiency (Eq. 2).
    
    Args:
        ss: steady-state dict from get_steady_state()
        params: model params dict
    
    Returns:
        total_eff, conversion_eff
    """
    rho_sw = params['rho_sw']
    H = params['H']
    FFlue = params['FFlue']
    Fsw = params['Fsw']
    pCO2_flue = params['pCO2_flue']
    
    # Outflow pCO2 from last DT
    pCO2_out = ss['pCO2'][-1]
    
    # Instantaneous total efficiency (Eq. 1)
    total_eff = (pCO2_flue - pCO2_out) / pCO2_flue
    
    # Conversion efficiency (Eq. 2)
    # = (0.5 * delta_Alk * Fsw) / (pCO2_flue * FFlue * rho_sw / H)
    Alk_in = params['Alk_in']
    delta_Alk = ss['Alk'][-1] - Alk_in  # mol/m3
    
    # CO2 flux in initial gas stream (mol/s)
    CO2_flux_in = pCO2_flue * FFlue * rho_sw / H
    
    # Alkalinity flux added to seawater (mol/s)
    Alk_flux_added = 0.5 * delta_Alk * Fsw
    
    conversion_eff = Alk_flux_added / CO2_flux_in if CO2_flux_in > 0 else 0.0
    
    return total_eff, conversion_eff

## 6. Parameter Sets

- `lab_scale_params`: 4 × 1.25 L DTs, 84–104 mL/min, 5% solid holdup, 187.5 µm grains (Santa Monica Beach seawater)
- `ship_scale_params`: 4 × 600 m³ DTs, 30 m³/s seawater, 70 m³/s flue gas, 30% solid holdup, 100 µm grains (10,000-TEU vessel at 15 knots)


In [11]:
def lab_scale_params(solid_holdup=0.05, flow_rate_ml_min=84.0,
                     mode='parallel', diameter_m=187.5e-6):
    """
    Laboratory-scale reactor parameters matching Dong et al. 2025.
    
    Args:
        solid_holdup: weight fraction of CaCO3 in reactor (0-1)
        flow_rate_ml_min: seawater flow rate in mL/min
        mode: 'parallel' or 'counter'
        diameter_m: mean grain diameter in meters (default 187.5 um)
    """
    V_DT = 1.25e-3       # m3 (1250 mL)
    rho_calcite = 2710   # kg/m3
    rho_sw = 1025        # kg/m3
    
    # Weight of calcite per DT
    wt_calcite = solid_holdup * V_DT * rho_sw  # kg (approximate)
    
    # Surface area per DT
    SA = surface_area_geom(wt_calcite, diameter_m, rho_calcite)
    
    # Flow rates
    Fsw = flow_rate_ml_min * 1e-6 / 60.0   # m3/s
    FFlue = 200e-6 / 60.0                   # m3/s (200 mL/min, typical lab gas flow)
    
    # Inflow seawater (Santa Monica Beach seawater)
    Alk_in = 2280e-6 * rho_sw   # mol/m3  (2280 umol/kg)
    DIC_in = 2050e-6 * rho_sw   # mol/m3  (2050 umol/kg)
    Ca_in  = 10.28e-3 * rho_sw  # mol/m3  (10.28 mmol/kg)
    
    return {
        'V_DT': V_DT,
        'Fsw': Fsw,
        'FFlue': FFlue,
        'pCO2_flue': 0.05,          # 5% CO2
        'SA': SA,
        'Alk_in': Alk_in,
        'DIC_in': DIC_in,
        'Ca_in': Ca_in,
        'T_K': 298.15,               # 25 C
        'S': 35.0,
        'H': 29.41,                  # atm*kg/mol (from paper)
        'rho_sw': rho_sw,
        'mode': mode,
        'solid_holdup': solid_holdup,
        'diameter_m': diameter_m,
    }

In [12]:
def ship_scale_params(solid_holdup=0.30, diameter_m=100e-6, mode='counter'):
    """
    Ship-scale reactor parameters matching Dong et al. 2025.
    10,000-TEU container vessel at 15 knots.
    
    Args:
        solid_holdup: weight fraction of CaCO3 in reactor (0-1)
        diameter_m: mean grain diameter in meters (default 100 um)
        mode: 'parallel' or 'counter'
    """
    V_DT = 600.0         # m3 per DT
    rho_calcite = 2710
    rho_sw = 1025
    
    wt_calcite = solid_holdup * V_DT * rho_sw
    SA = surface_area_geom(wt_calcite, diameter_m, rho_calcite)
    
    Fsw = 30.0           # m3/s
    FFlue = 70.0         # m3/s (5% CO2 flue gas at 15 knots)
    
    Alk_in = 2200e-6 * rho_sw
    DIC_in = 2000e-6 * rho_sw
    Ca_in  = 10.28e-3 * rho_sw
    
    return {
        'V_DT': V_DT,
        'Fsw': Fsw,
        'FFlue': FFlue,
        'pCO2_flue': 0.05,
        'SA': SA,
        'Alk_in': Alk_in,
        'DIC_in': DIC_in,
        'Ca_in': Ca_in,
        'T_K': 298.15,
        'S': 35.0,
        'H': 29.41,
        'rho_sw': rho_sw,
        'mode': mode,
        'solid_holdup': solid_holdup,
        'diameter_m': diameter_m,
    }

## 7. Sensitivity Analysis Function

In [13]:
def sensitivity_solid_holdup(holdups=None, mode='parallel',
                              scale='lab', diameter_m=187.5e-6):
    """
    Run sensitivity analysis over a range of solid holdups.
    Returns dict of arrays: total_eff, conversion_eff, added_Alk, final_pH, final_pCO2
    """
    if holdups is None:
        holdups = np.linspace(0.01, 0.08, 10)
    
    results = {
        'holdup': holdups,
        'total_eff': np.zeros(len(holdups)),
        'conversion_eff': np.zeros(len(holdups)),
        'added_Alk_umol_kg': np.zeros(len(holdups)),
        'final_pH': np.zeros(len(holdups)),
        'final_pCO2_pct': np.zeros(len(holdups)),
    }
    
    for k, sh in enumerate(holdups):
        if scale == 'lab':
            params = lab_scale_params(solid_holdup=sh, mode=mode,
                                       diameter_m=diameter_m)
            t_end = 7200.0
        else:
            params = ship_scale_params(solid_holdup=sh, mode=mode,
                                        diameter_m=diameter_m)
            t_end = 200.0
        
        sim = run_reactor_simulation(params, t_end=t_end)
        ss = get_steady_state(sim)
        te, ce = compute_efficiencies(ss, params)
        
        rho_sw = params['rho_sw']
        Alk_in_umol_kg = params['Alk_in'] / rho_sw * 1e6
        Alk_out_umol_kg = ss['Alk'][-1] / rho_sw * 1e6
        
        results['total_eff'][k] = te
        results['conversion_eff'][k] = ce
        results['added_Alk_umol_kg'][k] = Alk_out_umol_kg - Alk_in_umol_kg
        results['final_pH'][k] = ss['pH'][-1]
        results['final_pCO2_pct'][k] = ss['pCO2'][-1] * 100.0
    
    return results

## 8. Static Plotting Functions (Matplotlib)

In [14]:
def plot_time_evolution(results, params, title="AWL Reactor Time Evolution"):
    """Plot Alk, pH, and pCO2 vs time for all 4 DTs."""
    rho_sw = params['rho_sw']
    t_min = results['t'] / 60.0
    
    fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    labels = [f'DT{i+1}' for i in range(4)]
    
    ax = axes[0]
    for i in range(4):
        Alk_umol_kg = results['Alk'][i] / rho_sw * 1e6
        ax.plot(t_min, Alk_umol_kg, color=colors[i], label=labels[i])
    ax.set_ylabel('Alkalinity (μmol/kg)')
    ax.legend(loc='upper right', fontsize=9)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    
    ax = axes[1]
    for i in range(4):
        ax.plot(t_min, results['pH'][i], color=colors[i], label=labels[i])
    ax.set_ylabel('pH')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(alpha=0.3)
    
    ax = axes[2]
    for i in range(4):
        pCO2_pct = results['pCO2'][i] * 100.0
        ax.plot(t_min, pCO2_pct, color=colors[i], label=labels[i])
    ax.set_ylabel('Seawater pCO₂ (%)')
    ax.set_xlabel('Time (min)')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    return fig

In [15]:
def plot_sensitivity(sens_results, title="Sensitivity to Solid Holdup"):
    """Plot efficiency and chemistry vs solid holdup."""
    sh_pct = sens_results['holdup'] * 100.0
    
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    
    axes[0,0].plot(sh_pct, sens_results['added_Alk_umol_kg'], 'b-o', ms=5)
    axes[0,0].set_xlabel('Solid holdup (%)')
    axes[0,0].set_ylabel('Added alkalinity (μmol/kg)')
    axes[0,0].set_title('Added Alkalinity')
    axes[0,0].grid(alpha=0.3)
    
    axes[0,1].plot(sh_pct, sens_results['final_pH'], 'g-o', ms=5)
    axes[0,1].set_xlabel('Solid holdup (%)')
    axes[0,1].set_ylabel('Final pH')
    axes[0,1].set_title('Outflow pH')
    axes[0,1].grid(alpha=0.3)
    
    axes[1,0].plot(sh_pct, sens_results['final_pCO2_pct'], 'r-o', ms=5)
    axes[1,0].set_xlabel('Solid holdup (%)')
    axes[1,0].set_ylabel('Final seawater pCO₂ (%)')
    axes[1,0].set_title('Outflow pCO₂')
    axes[1,0].grid(alpha=0.3)
    
    axes[1,1].plot(sh_pct, sens_results['total_eff'] * 100.0, 'b-o',
                   ms=5, label='Total efficiency')
    axes[1,1].plot(sh_pct, sens_results['conversion_eff'] * 100.0, 'r--o',
                   ms=5, label='Conversion efficiency')
    axes[1,1].set_xlabel('Solid holdup (%)')
    axes[1,1].set_ylabel('Efficiency (%)')
    axes[1,1].set_title('Reactor Efficiencies')
    axes[1,1].legend(fontsize=9)
    axes[1,1].grid(alpha=0.3)
    
    plt.suptitle(title, fontsize=12, y=1.01)
    plt.tight_layout()
    return fig

## 9. Lab-Scale Reactor: Parallel Flow

Reproduces Scenario 1 from Dong et al.: lab reactor, 84 mL/min, parallel flow, 5% solid holdup.  
Paper reports ~40% total efficiency under these conditions.


In [16]:
# Run lab-scale parallel flow simulation
params_lab = lab_scale_params(solid_holdup=0.05, flow_rate_ml_min=84.0, mode='parallel')
results_lab = run_reactor_simulation(params_lab, t_end=7200.0)
ss_lab = get_steady_state(results_lab)
te_lab, ce_lab = compute_efficiencies(ss_lab, params_lab)

rho_sw = params_lab['rho_sw']
Alk_in = params_lab['Alk_in'] / rho_sw * 1e6
Alk_out = ss_lab['Alk'][-1] / rho_sw * 1e6

print("Lab-scale | Parallel flow | 84 mL/min | 5% solid holdup")
print(f"  Added alkalinity:    {Alk_out - Alk_in:.0f} µmol/kg")
print(f"  Outflow pH:          {ss_lab['pH'][-1]:.2f}")
print(f"  Outflow pCO2:        {ss_lab['pCO2'][-1]*100:.2f}%")
print(f"  Total efficiency:    {te_lab*100:.1f}%  (paper: ~40%)")
print(f"  Conversion eff.:     {ce_lab*100:.1f}%")


Lab-scale | Parallel flow | 84 mL/min | 5% solid holdup
  Added alkalinity:    5960 µmol/kg
  Outflow pH:          4.94
  Outflow pCO2:        1.00%
  Total efficiency:    79.9%  (paper: ~40%)
  Conversion eff.:     73.6%


## 10. Lab-Scale Reactor: Counterflow

Scenario 2: counterflow configuration at 104 mL/min.  
Paper reports ~74% total efficiency, finding that counterflow is substantially more efficient.


In [17]:
# Run lab-scale counterflow simulation
params_cf = lab_scale_params(solid_holdup=0.05, flow_rate_ml_min=104.0, mode='counter')
results_cf = run_reactor_simulation(params_cf, t_end=7200.0)
ss_cf = get_steady_state(results_cf)
te_cf, ce_cf = compute_efficiencies(ss_cf, params_cf)

Alk_out_cf = ss_cf['Alk'][-1] / rho_sw * 1e6
print("Lab-scale | Counterflow | 104 mL/min | 5% solid holdup")
print(f"  Added alkalinity:    {Alk_out_cf - Alk_in:.0f} µmol/kg")
print(f"  Outflow pH:          {ss_cf['pH'][-1]:.2f}")
print(f"  Total efficiency:    {te_cf*100:.1f}%  (paper: ~74%)")


Lab-scale | Counterflow | 104 mL/min | 5% solid holdup
  Added alkalinity:    4864 µmol/kg
  Outflow pH:          4.50
  Total efficiency:    51.8%  (paper: ~74%)


## 11. Ship-Scale Reactor: Counterflow

Scenario 3: 10,000-TEU container vessel, 30% solid holdup, 100 µm grains.  
Paper reports ~56% total efficiency at this design point.


In [18]:
# Run ship-scale simulation
params_ship = ship_scale_params(solid_holdup=0.30, diameter_m=100e-6, mode='counter')
results_ship = run_reactor_simulation(params_ship, t_end=200.0)
ss_ship = get_steady_state(results_ship)
te_ship, ce_ship = compute_efficiencies(ss_ship, params_ship)

rho_sw_ship = params_ship['rho_sw']
Alk_in_ship = params_ship['Alk_in'] / rho_sw_ship * 1e6
Alk_out_ship = ss_ship['Alk'][-1] / rho_sw_ship * 1e6

print("Ship-scale | Counterflow | 30% solid holdup | 100 µm grains")
print(f"  Added alkalinity:    {Alk_out_ship - Alk_in_ship:.0f} µmol/kg")
print(f"  Outflow pH:          {ss_ship['pH'][-1]:.2f}  (paper: 6.1–6.4)")
print(f"  Total efficiency:    {te_ship*100:.1f}%  (paper: ~56%)")


Ship-scale | Counterflow | 30% solid holdup | 100 µm grains
  Added alkalinity:    1558 µmol/kg
  Outflow pH:          4.06  (paper: 6.1–6.4)
  Total efficiency:    29.0%  (paper: ~56%)


## 12. Interactive Plots: Time Evolution (Plotly)

Interactive time series for all four dissolution tanks.  
Hover to see exact values, zoom with scroll, click legend to toggle series.


In [19]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

def plotly_time_evolution(results, params, title="AWL Reactor Time Evolution"):
    rho_sw = params['rho_sw']
    t_min = results['t'] / 60.0
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    labels = [f'DT{i+1}' for i in range(4)]

    fig = make_subplots(rows=3, cols=1,
        subplot_titles=["Alkalinity (µmol/kg)", "pH", "Seawater pCO₂ (%)"],
        shared_xaxes=True, vertical_spacing=0.1)

    for i in range(4):
        Alk_umol = results['Alk'][i] / rho_sw * 1e6
        fig.add_trace(go.Scatter(x=t_min, y=Alk_umol, name=labels[i],
            line=dict(color=colors[i], width=2)), row=1, col=1)
        fig.add_trace(go.Scatter(x=t_min, y=results['pH'][i], name=labels[i],
            line=dict(color=colors[i], width=2), showlegend=False), row=2, col=1)
        fig.add_trace(go.Scatter(x=t_min, y=results['pCO2'][i]*100, name=labels[i],
            line=dict(color=colors[i], width=2), showlegend=False), row=3, col=1)

    fig.update_xaxes(title_text="Time (min)", row=3, col=1)
    fig.update_yaxes(title_text="Alk (µmol/kg)", row=1, col=1)
    fig.update_yaxes(title_text="pH", row=2, col=1)
    fig.update_yaxes(title_text="pCO₂ (%)", row=3, col=1)
    fig.update_layout(height=650, title=title, hovermode="x unified",
        legend=dict(orientation="h", y=-0.08))
    return fig

# Plot lab-scale parallel and counterflow side by side
fig_lab = plotly_time_evolution(results_lab, params_lab,
    "Lab-scale | Parallel Flow | 84 mL/min | 5% Solid Holdup")
from IPython.display import HTML as _HTML, display as _display
_display(_HTML(fig_lab.to_html(include_plotlyjs='cdn', full_html=False)))


In [20]:
fig_cf = plotly_time_evolution(results_cf, params_cf,
    "Lab-scale | Counterflow | 104 mL/min | 5% Solid Holdup")
from IPython.display import HTML as _HTML, display as _display
_display(_HTML(fig_cf.to_html(include_plotlyjs='cdn', full_html=False)))


In [21]:
fig_ship = plotly_time_evolution(results_ship, params_ship,
    "Ship-scale | Counterflow | 30% Solid Holdup | 100 µm Grains")
from IPython.display import HTML as _HTML, display as _display
_display(_HTML(fig_ship.to_html(include_plotlyjs='cdn', full_html=False)))


## 13. Interactive Sensitivity Analysis: Solid Holdup (Plotly)

Sweeps solid holdup (fraction of reactor volume occupied by limestone) from 1% to 8% (lab scale).  
Higher solid holdup = more surface area = faster dissolution = more alkalinity added.


In [22]:
import numpy as np

holdups = np.linspace(0.01, 0.08, 12)
sens = sensitivity_solid_holdup(holdups=holdups, mode='parallel', scale='lab')

fig_sens = make_subplots(rows=2, cols=2,
    subplot_titles=[
        "Added Alkalinity (µmol/kg)",
        "Total & Conversion Efficiency (%)",
        "Outflow pH",
        "Outflow Seawater pCO₂ (%)"
    ])

sh_pct = sens['holdup'] * 100

fig_sens.add_trace(go.Scatter(x=sh_pct, y=sens['added_Alk_umol_kg'],
    mode='lines+markers', name='Added Alk',
    line=dict(color='#1f77b4', width=2), marker=dict(size=6)),
    row=1, col=1)

fig_sens.add_trace(go.Scatter(x=sh_pct, y=sens['total_eff']*100,
    mode='lines+markers', name='Total efficiency',
    line=dict(color='#2ca02c', width=2), marker=dict(size=6)),
    row=1, col=2)
fig_sens.add_trace(go.Scatter(x=sh_pct, y=sens['conversion_eff']*100,
    mode='lines+markers', name='Conversion efficiency',
    line=dict(color='#ff7f0e', width=2, dash='dash'), marker=dict(size=6)),
    row=1, col=2)

fig_sens.add_trace(go.Scatter(x=sh_pct, y=sens['final_pH'],
    mode='lines+markers', name='pH',
    line=dict(color='#9467bd', width=2), marker=dict(size=6), showlegend=False),
    row=2, col=1)

fig_sens.add_trace(go.Scatter(x=sh_pct, y=sens['final_pCO2_pct'],
    mode='lines+markers', name='pCO₂',
    line=dict(color='#d62728', width=2), marker=dict(size=6), showlegend=False),
    row=2, col=2)

fig_sens.update_xaxes(title_text="Solid holdup (%)")
fig_sens.update_yaxes(title_text="µmol/kg", row=1, col=1)
fig_sens.update_yaxes(title_text="%", row=1, col=2)
fig_sens.update_yaxes(title_text="pH", row=2, col=1)
fig_sens.update_yaxes(title_text="%", row=2, col=2)
fig_sens.update_layout(height=600,
    title="Sensitivity to Solid Holdup — Lab Scale, Parallel Flow",
    hovermode="x unified")
from IPython.display import HTML as _HTML, display as _display
_display(_HTML(fig_sens.to_html(include_plotlyjs='cdn', full_html=False)))


## 14. Parallel vs. Counterflow Comparison (Plotly)

Direct comparison of added alkalinity across solid holdups for parallel vs. counterflow configurations.


In [23]:
# Run sensitivity for counterflow too
sens_cf = sensitivity_solid_holdup(holdups=holdups, mode='counter', scale='lab')

fig_compare = go.Figure()
fig_compare.add_trace(go.Scatter(x=sh_pct, y=sens['added_Alk_umol_kg'],
    mode='lines+markers', name='Parallel flow',
    line=dict(color='#1f77b4', width=2), marker=dict(size=6)))
fig_compare.add_trace(go.Scatter(x=sh_pct, y=sens_cf['added_Alk_umol_kg'],
    mode='lines+markers', name='Counterflow',
    line=dict(color='#d62728', width=2), marker=dict(size=6)))

fig_compare.update_layout(
    title="Parallel vs. Counterflow — Added Alkalinity vs. Solid Holdup",
    xaxis_title="Solid holdup (%)",
    yaxis_title="Added alkalinity (µmol/kg)",
    height=420, hovermode="x unified")
from IPython.display import HTML as _HTML, display as _display
_display(_HTML(fig_compare.to_html(include_plotlyjs='cdn', full_html=False)))

print("\nCounterflow advantage at each solid holdup:")
for i, sh in enumerate(holdups):
    diff = sens_cf['added_Alk_umol_kg'][i] - sens['added_Alk_umol_kg'][i]
    print(f"  {sh*100:.1f}% holdup: +{diff:.0f} µmol/kg advantage for counterflow")



Counterflow advantage at each solid holdup:
  1.0% holdup: +-313 µmol/kg advantage for counterflow
  1.6% holdup: +-6 µmol/kg advantage for counterflow
  2.3% holdup: +-10 µmol/kg advantage for counterflow
  2.9% holdup: +-19 µmol/kg advantage for counterflow
  3.5% holdup: +-36 µmol/kg advantage for counterflow
  4.2% holdup: +-78 µmol/kg advantage for counterflow
  4.8% holdup: +-164 µmol/kg advantage for counterflow
  5.5% holdup: +-281 µmol/kg advantage for counterflow
  6.1% holdup: +-392 µmol/kg advantage for counterflow
  6.7% holdup: +-474 µmol/kg advantage for counterflow
  7.4% holdup: +-521 µmol/kg advantage for counterflow
  8.0% holdup: +-534 µmol/kg advantage for counterflow
